# Benefit Year Rollover and Cross-Year Edge Cases

At the start of every benefit year, each member's deductible and OOPM
balances should reset to zero. For most plans this happens on January 1.

When the reset doesn't happen, the member starts the new year carrying
last year's balance. If they had $2,800 applied toward a $3,000 deductible
in 2025, the system thinks they only need $200 more. The plan starts paying
at 80% (or whatever the coinsurance is) after the member's first fill in
January, when they should still owe the full $3,000.

There are also claims that straddle the year boundary - a prescription filled
on December 28 but not processed until January 3. If the processing pipeline
keys off processing_date instead of service_date, those claims get attributed
to the wrong benefit year.

And then there are plans that don't use a calendar year at all. A municipal
plan with an October 1 fiscal year start needs its accumulators to reset on
October 1, not January 1. If the rollover automation only knows about Jan 1,
these plans never get a proper reset.

This notebook checks for all three rollover-related failure modes:

| Check | What it catches |
|-------|----------------|
| Rollover failures | Active members with 2025 records but no 2026 record initialized |
| Cross-year claims | Claims with 2025 service dates incorrectly tagged as 2026 benefit year |
| Non-calendar plans | Plans with fiscal year starts that the standard Jan 1 rollover doesn't cover |

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261019_115429


In [0]:
from pyspark.sql import functions as F

spark.sql("USE pbm_sim")

NB = "NB_05"

# Pre-flight- we need accumulators from both systems, claims for the
# reprocessing exposure check, plan_design for the non-calendar benefit
# year start months, and members to distinguish true rollover failures
# from expected terminations.
for tbl in ["pbm_sim.pbm_accumulators", "pbm_sim.tpa_accumulator_feed",
            "pbm_sim.pharmacy_claims", "pbm_sim.plan_design", "pbm_sim.members"]:
    validate_table_exists(tbl, NB)

df_pbm_accum = spark.table("pbm_sim.pbm_accumulators")
df_tpa_accum = spark.table("pbm_sim.tpa_accumulator_feed")
df_claims = spark.table("pbm_sim.pharmacy_claims")
df_plan = spark.table("pbm_sim.plan_design")
df_members = spark.table("pbm_sim.members")

log_entry(NB, "load_source_tables", "SUCCESS")

# Separate PBM accumulators by benefit year
df_accum_2025 = df_pbm_accum.filter(F.col("benefit_year") == 2025)
df_accum_2026 = df_pbm_accum.filter(F.col("benefit_year") == 2026)

print(f"PBM accumulator records (2025): {df_accum_2025.count()}")
print(f"PBM accumulator records (2026): {df_accum_2026.count()}")

[PIPELINE_LOG] {"run_id": "RUN_20261019_115429", "timestamp": "2026-10-19T11:54:30.717674+00:00", "notebook": "NB_05", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}
PBM accumulator records (2025): 2375
PBM accumulator records (2026): 45177


## 1. Rollover Failures

A member who had accumulator activity in 2025 should have a fresh record
initialized for 2026 with balances at zero. If that 2026 record doesn't
exist, one of two things happened:

- The rollover job skipped them, or
- They were terminated before 2026 started.

Terminated members are expected to be missing. Active members with no
2026 record are the problem - the rollover automation didn't pick them up.

In [0]:
# Members with a 2025 accumulator record
df_had_2025 = df_accum_2025.select("member_id", "plan_id").distinct()

# Members with a 2026 accumulator record
df_has_2026 = df_accum_2026.select("member_id").distinct()

# left_anti gives us the 2025 members who have NO matching 2026 record
df_missing_2026 = df_had_2025.join(df_has_2026, on="member_id", how="left_anti")
missing_count = df_missing_2026.count()

print(f"Members with 2025 record but no 2026 record: {missing_count}")

Members with 2025 record but no 2026 record: 375


In [0]:
# Now filter out the ones who are legitimately terminated.
# If the member's term_date is before Jan 1 2026, they shouldn't
# have a 2026 record - that's expected, not a failure.

df_rollover_check = (
    df_missing_2026
    .join(df_members.select("member_id", "term_date", "effective_date"), on="member_id")
    .withColumn(
        "status_on_jan1",
        F.when(
            F.col("term_date").isNull() | (F.col("term_date") >= F.lit("2026-01-01")),
            "ACTIVE_OR_TERMINATED_IN_2026"
        ).otherwise("TERMINATED_BEFORE_2026")
    )
)

# df_true_failures is the set we care about - active members the rollover job missed.
# We'll use this later to check for claims that may need reprocessing.
df_true_failures = df_rollover_check.filter(F.col("status_on_jan1") == "ACTIVE_OR_TERMINATED_IN_2026")
df_expected_missing = df_rollover_check.filter(F.col("status_on_jan1") == "TERMINATED_BEFORE_2026")

true_failure_count = df_true_failures.count()
expected_missing_count = df_expected_missing.count()

print(f"Rollover failures (active or termed in 2026, no 2026 record): {true_failure_count}")
print(f"Expected missing (terminated before 2026):                    {expected_missing_count}")

log_entry(NB, "rollover_failures", "SUCCESS",
          detail=f"true_failures={true_failure_count}, expected_missing={expected_missing_count}")

Rollover failures (active or termed in 2026, no 2026 record): 375
Expected missing (terminated before 2026):                    0
[PIPELINE_LOG] {"run_id": "RUN_20261019_115429", "timestamp": "2026-10-19T11:54:41.632845+00:00", "notebook": "NB_05", "section": "rollover_failures", "status": "SUCCESS", "row_count": 0, "detail": "true_failures=375, expected_missing=0", "duration_sec": 0.0}


In [0]:
# Show the rollover failure detail with their 2025 balances so the
# operations team can see who's carrying stale data.
if true_failure_count > 0:
    print("Active members whose accumulators failed to roll over:")
    df_failure_detail = (
        df_true_failures
        .join(
            df_accum_2025.select(
                "member_id",
                F.col("pharmacy_deductible_applied").alias("ded_2025"),
                F.col("pharmacy_oopm_applied").alias("oopm_2025"),
                F.col("last_claim_date").alias("last_claim_2025"),
            ),
            on="member_id"
        )
        .select(
            "member_id", "plan_id", "effective_date", "term_date",
            "ded_2025", "oopm_2025", "last_claim_2025"
        )
        .orderBy(F.col("ded_2025").desc())
    )
    display(df_failure_detail)

Active members whose accumulators failed to roll over:


member_id,plan_id,effective_date,term_date,ded_2025,oopm_2025,last_claim_2025
MBR1013582,PLN006,2026-01-01,null,3493.86,3861.68,2025-12-29
MBR1013331,PLN006,2026-01-01,null,3476.86,3709.0,2025-12-18
MBR1037333,PLN026,2026-01-01,null,3399.63,3808.06,2025-12-30
MBR1037076,PLN026,2026-01-01,null,3312.2,3428.35,2025-12-16
MBR1012921,PLN006,2026-01-01,null,3309.39,3684.78,2025-12-29
MBR1013707,PLN006,2026-01-01,null,3261.72,3426.78,2025-12-24
MBR1013578,PLN006,2026-01-01,null,3197.73,3589.53,2025-12-21
MBR1023357,PLN013,2026-01-01,null,3142.06,3407.23,2025-12-27
MBR1009406,PLN004,2026-01-01,null,3141.71,3425.58,2025-12-19
MBR1027268,PLN016,2026-01-01,null,3004.55,3372.32,2025-12-30


### Operational Impact of Rollover Failures

These members have no 2026 accumulator record on the PBM side. Depending
on how the adjudication engine handles a missing record, one of two things
is happening:

- The engine falls back to the most recent record it can find, which is the
  2025 record with last year's balances still on it. The member's 2025
  spending is being treated as 2026 spending, and they'll hit their
  deductible much sooner than they should.

- The engine treats a missing record as zero and creates one on the fly.
  The member is fine, but the TPA won't receive pharmacy accumulator data
  because the record the sync process reads from doesn't exist.

Either scenario needs correction.

In [0]:
# Check whether these members have any 2026 claims.
# If they do, those claims were adjudicated against whatever fallback
# logic the engine used, and they may need reprocessing.

if true_failure_count > 0:
    # Broadcast join instead of .collect() + .isin() to avoid pulling
    # all failure member_ids to the driver. At scale (millions of members),
    # the .collect() pattern causes driver OOM. Broadcasting the smaller
    # failure set lets Spark push the filter to the executors.
    df_failure_member_ids = F.broadcast(df_true_failures.select("member_id").distinct())

    df_claims_for_failures = (
        df_claims
        .join(df_failure_member_ids, on="member_id", how="inner")
        .filter(F.col("benefit_year") == 2026)
    )

    claims_affected = df_claims_for_failures.count()
    members_with_claims = df_claims_for_failures.select("member_id").distinct().count()
    dollars_at_risk = df_claims_for_failures.agg(
        F.sum("member_deductible_applied")
    ).collect()[0][0] or 0

    print(f"2026 claims for rollover-failure members: {claims_affected}")
    print(f"Of the {true_failure_count} affected members, {members_with_claims} have filled claims in 2026")
    print(f"Total deductible applied on those claims: ${dollars_at_risk:,.2f}")

    if claims_affected > 0:
        print(f"\nThese claims may have been adjudicated against stale 2025 accumulator data.")
    else:
        print(f"\nNo 2026 claims found - no reprocessing needed.")

    log_entry(NB, "rollover_claims_exposure", "SUCCESS",
              detail=f"claims_affected={claims_affected}, members_with_claims={members_with_claims}, dollars_at_risk={dollars_at_risk:.2f}")

2026 claims for rollover-failure members: 0
Of the 375 affected members, 0 have filled claims in 2026
Total deductible applied on those claims: $0.00

No 2026 claims found - no reprocessing needed.
[PIPELINE_LOG] {"run_id": "RUN_20261019_115429", "timestamp": "2026-10-19T11:55:14.250472+00:00", "notebook": "NB_05", "section": "rollover_claims_exposure", "status": "SUCCESS", "row_count": 0, "detail": "claims_affected=0, members_with_claims=0, dollars_at_risk=0.00", "duration_sec": 0.0}


## 2. Cross-Year Claims

A claim's benefit year should be determined by the service date, not the
processing date. If a prescription was filled on December 28, 2025 but
didn't process until January 3, 2026, it belongs to the 2025 benefit year.
The deductible and OOPM applied should count against 2025 limits.

When these claims are incorrectly tagged as benefit year 2026, the member's
2026 accumulator gets inflated by spending that already counted toward 2025,
and the 2025 final balances are understated.

In [0]:
df_cross_year = (
    df_claims
    .filter(
        (F.year(F.col("service_date")) == 2025) &
        (F.year(F.col("processing_date")) == 2026)
    )
)

cross_year_count = df_cross_year.count()
print(f"Claims with 2025 service date and 2026 processing date: {cross_year_count}")

Claims with 2025 service date and 2026 processing date: 40


In [0]:
# Check how many of these cross-year claims are tagged to the wrong
# benefit year. Correctly attributed means benefit_year = 2025 (keyed
# off service_date). Misattributed means benefit_year = 2026 (keyed
# off processing_date instead).

if cross_year_count > 0:
    df_misattributed = df_cross_year.filter(F.col("benefit_year") == 2026)
    df_correct = df_cross_year.filter(F.col("benefit_year") == 2025)

    misattributed_count = df_misattributed.count()
    correct_count = df_correct.count()

    print(f"Correctly attributed to 2025: {correct_count}")
    print(f"Incorrectly attributed to 2026: {misattributed_count}")

    if misattributed_count > 0:
        print(f"\nThese {misattributed_count} claims need their benefit_year corrected to 2025:")
        display(
            df_misattributed
            .select(
                "claim_id", "member_id", "plan_id",
                "service_date", "processing_date", "benefit_year",
                "drug_name", "drug_cost",
                "member_deductible_applied"
            )
            .orderBy("service_date")
        )

        misattributed_ded = df_misattributed.agg(
            F.sum("member_deductible_applied")
        ).collect()[0][0] or 0
        print(f"\nTotal deductible incorrectly counted against 2026: ${misattributed_ded:,.2f}")

    log_entry(NB, "cross_year_claims", "SUCCESS",
              detail=f"cross_year_total={cross_year_count}, correct={correct_count}, misattributed={misattributed_count}")
else:
    log_entry(NB, "cross_year_claims", "SUCCESS",
              detail="no cross-year claims found")

Correctly attributed to 2025: 0
Incorrectly attributed to 2026: 40

These 40 claims need their benefit_year corrected to 2025:


claim_id,member_id,plan_id,service_date,processing_date,benefit_year,drug_name,drug_cost,member_deductible_applied
CLM9000400005,MBR1023134,PLN013,2025-12-20,2026-01-08,2026,Sertraline 50mg,16.47,8.49
CLM9000400036,MBR1008624,PLN004,2025-12-20,2026-01-06,2026,Atorvastatin 20mg,50.69,12.61
CLM9000400009,MBR1040333,PLN029,2025-12-20,2026-01-02,2026,Amlodipine 5mg,23.28,13.91
CLM9000400007,MBR1017746,PLN009,2025-12-21,2026-01-03,2026,Amlodipine 5mg,28.62,0.37
CLM9000400027,MBR1035762,PLN024,2025-12-21,2026-01-07,2026,Lisinopril 10mg,29.76,11.04
CLM9000400011,MBR1007190,PLN003,2025-12-21,2026-01-09,2026,Omeprazole 20mg,31.74,2.26
CLM9000400010,MBR1007479,PLN004,2025-12-23,2026-01-05,2026,Prednisone 10mg,10.5,16.08
CLM9000400013,MBR1043165,PLN032,2025-12-23,2026-01-07,2026,Losartan 50mg,32.15,14.44
CLM9000400028,MBR1007519,PLN004,2025-12-24,2026-01-02,2026,Prednisone 10mg,11.12,10.48
CLM9000400001,MBR1033286,PLN022,2025-12-24,2026-01-04,2026,Lisinopril 10mg,26.93,5.27



Total deductible incorrectly counted against 2026: $362.29
[PIPELINE_LOG] {"run_id": "RUN_20261019_115429", "timestamp": "2026-10-19T11:55:40.763705+00:00", "notebook": "NB_05", "section": "cross_year_claims", "status": "SUCCESS", "row_count": 0, "detail": "cross_year_total=40, correct=0, misattributed=40", "duration_sec": 0.0}


## 3. Non-Calendar Benefit Year Plans

Most employer plans run on a calendar year (Jan 1 to Dec 31). But some
plans use a fiscal year that starts on a different month.

For these plans, the accumulator reset should happen on their fiscal year
start, not January 1. If the rollover job only knows about Jan 1 boundaries,
these members' accumulators won't reset on time. Conversely, they should
NOT reset on January 1 like everyone else.

This is one of the more subtle configuration issues because it's not a data
error. The data is technically correct. The problem is whether the rollover
automation accounts for non-standard benefit year start dates.

In [0]:
df_non_calendar = df_plan.filter(F.col("benefit_year_start_month") != 1)

print("Plans with non-calendar benefit years:")
display(
    df_non_calendar
    .select(
        "plan_id", "plan_name", "benefit_year_start_month",
        "individual_deductible", "individual_oopm"
    )
)


Plans with non-calendar benefit years:


plan_id,plan_name,benefit_year_start_month,individual_deductible,individual_oopm
PLN007,Eta Municipal Workers,10,2000,7000
PLN038,Northstar Aviation,7,2400,7400
PLN039,Orion Chemical Works,4,2200,7000


In [0]:
# Approach: join non-calendar plans to accumulators and claims
# in bulk instead of looping with per-plan .collect() + .filter() queries.
# This pushes all the work to the executors in two passes rather than
# 2*N passes (one accumulator query + one claims query per plan).

# -- Accumulator distribution for all non-calendar plans at once --
df_nc_accum_summary = (
    df_pbm_accum
    .join(df_non_calendar.select("plan_id", "plan_name", "benefit_year_start_month"), on="plan_id")
    .groupBy("plan_id", "plan_name", "benefit_year_start_month", "benefit_year")
    .agg(
        F.count("*").alias("records"),
        F.round(F.avg("pharmacy_deductible_applied"), 2).alias("avg_ded"),
    )
    .orderBy("plan_id", "benefit_year")
)

print("Non-calendar plan accumulator records by benefit year:")
display(df_nc_accum_summary)

Non-calendar plan accumulator records by benefit year:


plan_id,plan_name,benefit_year_start_month,benefit_year,records,avg_ded
PLN007,Eta Municipal Workers,10,2025,64,1246.26
PLN007,Eta Municipal Workers,10,2026,1309,453.75
PLN038,Northstar Aviation,7,2025,28,1578.24
PLN038,Northstar Aviation,7,2026,657,516.87
PLN039,Orion Chemical Works,4,2025,28,1367.14
PLN039,Orion Chemical Works,4,2026,632,515.91


In [0]:
# -- Benefit year misattribution check for all non-calendar plans at once --
# For a plan with an October 1 start, a claim on November 15 2025 belongs
# to the Oct 2025 - Sep 2026 benefit year. The expected_benefit_year logic
# below uses each plan's specific start month to compute the correct year,
# then flags claims where the recorded benefit_year doesn't match.

df_nc_claims_check = (
    df_claims
    .join(
        df_non_calendar.select("plan_id", "plan_name", "benefit_year_start_month"),
        on="plan_id"
    )
    .withColumn(
        "fiscal_year_start",
        F.make_date(
            F.when(
                F.month(F.col("service_date")) >= F.col("benefit_year_start_month"),
                F.year(F.col("service_date"))
            ).otherwise(F.year(F.col("service_date")) - 1),
            F.col("benefit_year_start_month"),
            F.lit(1)
        )
    )
    .withColumn(
        "expected_benefit_year",
        F.when(
            F.col("service_date") >= F.col("fiscal_year_start"),
            F.year(F.col("fiscal_year_start")) + 1  # e.g., Oct 2025 start -> BY 2026
        ).otherwise(F.year(F.col("fiscal_year_start")))
    )
    .filter(F.col("benefit_year") != F.col("expected_benefit_year"))
)

nc_mismatch_count = df_nc_claims_check.count()

if nc_mismatch_count > 0:
    print(f"\nClaims with benefit_year mismatch for non-calendar fiscal year logic: {nc_mismatch_count}")
    display(
        df_nc_claims_check
        .select(
            "plan_id", "plan_name", "benefit_year_start_month",
            "claim_id", "service_date", "benefit_year", "expected_benefit_year"
        )
        .orderBy("plan_id", "service_date")
        .limit(50)
    )
else:
    print("\nNo benefit_year attribution mismatches found for non-calendar plans.")

# Per-plan configuration note - small cardinality, safe to collect
for plan_row in df_non_calendar.collect():
    pid = plan_row["plan_id"]
    pname = plan_row["plan_name"]
    start_month = plan_row["benefit_year_start_month"]
    print(f"\n  NOTE: Confirm that the rollover automation has a specific rule")
    print(f"  for {pid} ({pname}) to reset accumulators on month {start_month}")
    print(f"  instead of Jan 1. A Jan 1 reset would prematurely zero out")
    print(f"  mid-benefit-year balances.")

log_entry(NB, "non_calendar_plans", "SUCCESS",
          detail=f"non_calendar_count={df_non_calendar.count()}, misattributed_claims={nc_mismatch_count}")


Claims with benefit_year mismatch for non-calendar fiscal year logic: 5693


plan_id,plan_name,benefit_year_start_month,claim_id,service_date,benefit_year,expected_benefit_year
PLN038,Northstar Aviation,7,CLM9000358902,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000159413,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000352118,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000022624,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000189451,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000357478,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000277418,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000056761,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000052383,2026-07-01,2026,2027
PLN038,Northstar Aviation,7,CLM9000236213,2026-07-01,2026,2027



  NOTE: Confirm that the rollover automation has a specific rule
  for PLN007 (Eta Municipal Workers) to reset accumulators on month 10
  instead of Jan 1. A Jan 1 reset would prematurely zero out
  mid-benefit-year balances.

  NOTE: Confirm that the rollover automation has a specific rule
  for PLN038 (Northstar Aviation) to reset accumulators on month 7
  instead of Jan 1. A Jan 1 reset would prematurely zero out
  mid-benefit-year balances.

  NOTE: Confirm that the rollover automation has a specific rule
  for PLN039 (Orion Chemical Works) to reset accumulators on month 4
  instead of Jan 1. A Jan 1 reset would prematurely zero out
  mid-benefit-year balances.
[PIPELINE_LOG] {"run_id": "RUN_20261019_115429", "timestamp": "2026-10-19T11:56:04.768002+00:00", "notebook": "NB_05", "section": "non_calendar_plans", "status": "SUCCESS", "row_count": 0, "detail": "non_calendar_count=3, misattributed_claims=5693", "duration_sec": 0.0}


## Summary

In [0]:
print(f"{'=' * 60}")
print(f"  BENEFIT YEAR ROLLOVER AUDIT RESULTS")
print(f"{'=' * 60}")
print(f"  Rollover failures (active, no 2026 record): {true_failure_count}")
if true_failure_count > 0:
    print(f"    2026 claims on affected members:          {claims_affected}")
    print(f"    Deductible at risk:                       ${dollars_at_risk:,.2f}")
print(f"{'=' * 60}")
if cross_year_count > 0:
    print(f"  Cross-year claims (2025 service, 2026 proc): {cross_year_count}")
    print(f"    Correctly attributed to 2025:               {correct_count}")
    print(f"    Incorrectly attributed to 2026:             {misattributed_count}")
    if misattributed_count > 0:
        print(f"    Deductible misattributed to 2026:           ${misattributed_ded:,.2f}")
print(f"{'=' * 60}")
nc_count = df_non_calendar.count()
print(f"  Non-calendar benefit year plans:              {nc_count}")
# Small cardinality, safe to collect for summary output
if nc_count > 0:
    for row in df_non_calendar.select("plan_id", "plan_name", "benefit_year_start_month").collect():
        print(f"    {row['plan_id']} ({row['plan_name']}): starts month {row['benefit_year_start_month']}")
print(f"{'=' * 60}")

  BENEFIT YEAR ROLLOVER AUDIT RESULTS
  Rollover failures (active, no 2026 record): 375
    2026 claims on affected members:          0
    Deductible at risk:                       $0.00
  Cross-year claims (2025 service, 2026 proc): 40
    Correctly attributed to 2025:               0
    Incorrectly attributed to 2026:             40
    Deductible misattributed to 2026:           $362.29
  Non-calendar benefit year plans:              3
    PLN007 (Eta Municipal Workers): starts month 10
    PLN038 (Northstar Aviation): starts month 7
    PLN039 (Orion Chemical Works): starts month 4


## Notes

**Rollover failures** are typically a batch job issue. The fix is to create
the missing 2026 records with zero balances and reprocess any claims that
were adjudicated in the interim. The harder part is figuring out why the
rollover job skipped these members. Common causes: the member was in a
pending enrollment status when the job ran, or the job used a member list
snapshot that was taken before these members were added.

**Cross-year claim misattribution** is a logic fix in the claims processing
pipeline. The benefit_year assignment should key off service_date, not
processing_date. If the pipeline already does this and these claims still
got tagged wrong, it could be a manual override or a retroactive adjustment
that bypassed the standard logic.

**Non-calendar benefit years** require plan-specific configuration in both
the rollover automation and the benefit_year attribution logic. The safest
approach is to derive the benefit year from the plan's
benefit_year_start_month and the claim's service_date at adjudication time,
rather than assuming calendar year boundaries.

In [0]:
log_summary(NB)